# Notebook 2.4: The Glauber-Sudarshan P-Representation and Operator Ordering
### Module 2: Phase Space and Quasiprobability Distributions
**Computational Quantum Optics Curriculum**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_2_phase_space_quasiprobabilities/04_glauber_sudarshan_p_representation.ipynb)

---

### Prerequisites
- Notebook 2.1: The Wigner Quasiprobability Distribution Fundamentals
- Notebook 2.2: Wigner Negativity as a Metric for Quantum Non-Classicality
- Notebook 2.3: The Husimi Q-Function and Heterodyne Measurement
- Notebook 1.6: The Mandel Q-Parameter and Non-Classicality Criteria

### Pedagogical Learning Objectives
1. **The Glauber-Sudarshan $P$-Representation:** Formulate $\hat{\rho} = \int P(\alpha) |\alpha\rangle\langle \alpha| \, d^2\alpha$, understanding its origin in normal-ordered expectation values $\langle (\hat{a}^\dagger)^m \hat{a}^n \rangle = \int P(\alpha) (\alpha^*)^m \alpha^n \, d^2\alpha$.
2. **Cahill-Glauber $s$-Parameterized Formalism:** Unify the three canonical phase-space distributions ($P, W, Q$) via the $s$-ordered characteristic function $\chi(\xi; s) = \mathrm{Tr}[\hat{\rho}\hat{D}(\xi)] e^{s|\xi|^2/2}$ with $s = +1$ (Glauber $P$), $s = 0$ (Wigner $W$), and $s = -1$ (Husimi $Q$).
3. **The Smoothing Hierarchy:** Mathematically trace the irreversible Gaussian deconvolution chain $P(\alpha) \to W(\alpha) \to Q(\alpha)$, revealing why $P$ contains the highest quantum information density.
4. **Singular Nature of Non-Classical $P$-Functions:** Understand why non-classical states (Fock states, squeezed states) possess $P$-distributions that cannot be ordinary functions, behaving as tempered distributions with derivatives of Dirac delta functions.
5. **Regularized $P$-Functions (Kiesel-Vogel Filter):** Implement the regularized non-classicality witness $P_\Omega(\alpha)$ via Fourier filtering, demonstrating observable negative $P_\Omega$ lobes for single-photon light.
6. **Numerical Synthesis & Interactive Parameter Sweep:** Build an end-to-end characteristic function solver that dynamically interpolates across the continuous ordering parameter $s \in [-1, +1]$.


In [ ]:
# Setup, environment detection, and library imports
import os
import sys
import math
import numpy as np
import scipy.linalg as la
from scipy.special import genlaguerre
import matplotlib.pyplot as plt

# Ensure matplotlib handles headless environments gracefully
if 'MPLBACKEND' not in os.environ:
    os.environ['MPLBACKEND'] = 'Agg' if not hasattr(sys, 'ps1') else 'module://matplotlib_inline.backend_inline'

# Plot styling configuration for high-DPI publication graphics
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2.0
plt.rcParams['grid.alpha'] = 0.35

# Detect QuTiP availability with clean fallback
try:
    import qutip as qt
    HAS_QUTIP = True
    print(f"QuTiP {qt.__version__} successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

# Detect interactive widgets environment
try:
    from ipywidgets import interact, FloatSlider, Dropdown
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


## 1. The Glauber-Sudarshan $P$-Representation

In 1963, **Roy J. Glauber** (*Phys. Rev. Lett.* 10, 84) and **E. C. George Sudarshan** (*Phys. Rev. Lett.* 10, 277) independently introduced a diagonal representation of the density operator in the overcomplete coherent state basis:

$$
\hat{\rho} = \int_{\mathbb{C}} P(\alpha) |\alpha\rangle\langle \alpha| \, d^2\alpha
$$

where $d^2\alpha = d(\text{Re}\alpha) \, d(\text{Im}\alpha)$.

---

### Physical Purpose: Normal-Ordered Expectation Values

The primary virtue of the $P$-representation is that **normal-ordered operator expectation values** translate into classical-like statistical averages:

$$
\langle (\hat{a}^\dagger)^m \hat{a}^n \rangle = \mathrm{Tr}\left[ \hat{\rho} (\hat{a}^\dagger)^m \hat{a}^n \right] = \int_{\mathbb{C}} P(\alpha) (\alpha^*)^m \alpha^n \, d^2\alpha
$$

Because photon detectors (photodiodes, PMTs) operate via photon absorption, photon counting measurements correspond directly to normal-ordered moments!

---

### The Strict Definition of Classical Light

If $P(\alpha)$ is a well-behaved, non-negative probability density:
$$
P(\alpha) \ge 0 \quad \text{and} \quad \int P(\alpha) \, d^2\alpha = 1
$$
then the quantum state $\hat{\rho}$ is an **incoherent statistical mixture of classical coherent states**. The field can be completely explained by classical electromagnetic wave theory with fluctuating complex amplitudes!

**The Quantum Frontier:**
If $P(\alpha)$ takes negative values, or is more singular than a Dirac delta function (e.g., derivatives of delta functions), the optical state **has no classical wave analogue**. It is fundamentally quantum!


## 2. Cahill-Glauber $s$-Parameterized Quasiprobability Hierarchy

In 1969, **K. E. Cahill and R. J. Glauber** (*Phys. Rev.* 177, 1857 & 1882) established a unified mathematical framework connecting all phase-space distributions via a single continuous ordering parameter $s \in [-1, +1]$:

---

### The $s$-Ordered Characteristic Function

Let $\hat{D}(\xi) = \exp(\xi \hat{a}^\dagger - \xi^* \hat{a})$ be the displacement operator. The $s$-parameterized characteristic function is:

$$
\chi(\xi; s) \equiv \mathrm{Tr}\left[ \hat{\rho} \hat{D}(\xi) \right] \exp\left( \frac{s}{2} |\xi|^2 \right)
$$

The phase-space distribution $W(\alpha; s)$ is the two-dimensional Fourier transform of $\chi(\xi; s)$:

$$
W(\alpha; s) = \frac{1}{\pi^2} \int_{\mathbb{C}} \chi(\xi; s) \exp\left( \alpha \xi^* - \alpha^* \xi \right) \, d^2\xi
$$

---

### The Holy Trinity of Quantum Phase Space:

| Parameter | Ordering Type | Distribution $W(\alpha; s)$ | Singularity / Negativity |
| :---: | :---: | :---: | :--- |
| **$s = +1$** | **Normal Ordering** | **Glauber-Sudarshan $P(\alpha)$** | Highly singular (derivatives of $\delta$), negative infinities |
| **$s = 0$** | **Symmetric (Weyl)** | **Wigner Function $W(\alpha)$** | Smooth, bounded $|W| \le 1/\pi$, can be negative |
| **$s = -1$** | **Anti-Normal** | **Husimi $Q$-Function $Q(\alpha)$** | Everywhere positive, smooth, $0 \le Q \le 1/\pi$ |

---

### The Continuous Smoothing Convolution

Because $\chi(\xi; s) = \chi(\xi; s') \exp\left( \frac{s - s'}{2} |\xi|^2 \right)$, transitioning from higher $s$ to lower $s$ in Fourier space is equivalent to **Gaussian convolution in phase space**:

$$
W(\alpha; s) = \frac{2}{\pi(s' - s)} \int_{\mathbb{C}} W(\beta; s') \exp\left( -\frac{2|\alpha - \beta|^2}{s' - s} \right) \, d^2\beta \quad (s < s')
$$

Setting $s' = +1$ and $s = 0$ gives: $P(\alpha) \xrightarrow{\ast \text{ Gauss}} W(\alpha)$.  
Setting $s' = 0$ and $s = -1$ gives: $W(\alpha) \xrightarrow{\ast \text{ Gauss}} Q(\alpha)$.


In [ ]:
# Comprehensive Numerical Solver for s-Parameterized Phase-Space Distributions

def characteristic_function(state, xi_grid):
    """Computes symmetric characteristic function chi(xi; s=0) = Tr[rho D(xi)] in Fock basis."""
    if state.ndim == 1:
        psi = state / np.linalg.norm(state)
        rho = np.outer(psi, psi.conj())
    else:
        rho = state / np.real(np.trace(state))
        
    N = rho.shape[0]
    M_xi = xi_grid.shape[0]
    chi = np.zeros_like(xi_grid, dtype=np.complex128)
    
    # Precompute factorial
    fact = np.array([float(math.factorial(n)) for n in range(N)])
    
    # Laguerre expansion for matrix elements <n|D(xi)|m>:
    # <n|D(xi)|m> = sqrt(n!/m!) * xi^(m-n) * exp(-|xi|^2 / 2) * L_n^(m-n)(|xi|^2)  (for m >= n)
    abs_xi2 = np.abs(xi_grid)**2
    exp_half = np.exp(-0.5 * abs_xi2)
    
    for n in range(N):
        if abs(rho[n, n]) > 1e-15:
            L_nn = genlaguerre(n, 0)(abs_xi2)
            chi += rho[n, n] * L_nn
            
        for m in range(n + 1, N):
            if abs(rho[n, m]) > 1e-15:
                d = m - n
                coeff = np.sqrt(fact[n] / fact[m]) * (xi_grid**d)
                L_nm = genlaguerre(n, d)(abs_xi2)
                term = coeff * L_nm
                chi += rho[n, m] * term + rho[m, n] * np.conj(term)
                
    return chi * exp_half

def compute_s_distribution(state, xvec, pvec, s=0.0):
    """Computes s-parameterized phase-space distribution via 2D Fourier transform."""
    # Create phase space grid
    X, P = np.meshgrid(xvec, pvec)
    alpha = (X + 1j * P) / np.sqrt(2.0)
    
    # Conjugate Fourier grid for xi
    dx = xvec[1] - xvec[0]
    dp = pvec[1] - pvec[0]
    N_pts = len(xvec)
    
    # Scale Fourier frequency domain
    dxi = np.pi / (xvec[-1] - xvec[0])
    xi_1d = np.fft.fftshift(np.fft.fftfreq(N_pts, d=dx/(2*np.pi)))
    Xi_x, Xi_p = np.meshgrid(xi_1d, xi_1d)
    Xi = (Xi_x + 1j * Xi_p) / np.sqrt(2.0)
    
    # 1. Compute chi(xi; s=0)
    chi_0 = characteristic_function(state, Xi)
    
    # 2. Scale by exp(s * |xi|^2 / 2)
    chi_s = chi_0 * np.exp(0.5 * s * np.abs(Xi)**2)
    
    # 3. 2D Inverse FFT to phase space
    W_s = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(chi_s)))
    W_s = np.real(W_s) / (dx * dp)
    
    # Normalize to unit integral
    tr_norm = np.sum(W_s) * dx * dp
    if abs(tr_norm) > 1e-6:
        W_s /= tr_norm
        
    return W_s

# Verification: Thermal state has well-behaved classical P-function
# P_thermal(alpha) = (1 / (pi * n_bar)) * exp(-|alpha|^2 / n_bar)
n_bar_val = 1.8
N_dim = 25
n_axis = np.arange(N_dim)
p_th_dist = (1.0 / (1.0 + n_bar_val)) * ((n_bar_val / (1.0 + n_bar_val))**n_axis)
rho_thermal = np.diag(p_th_dist).astype(np.complex128)

x_eval = np.linspace(-4.0, 4.0, 81)
p_eval = np.linspace(-4.0, 4.0, 81)

# Compute Wigner (s=0) and Husimi (s=-1)
W_th_s0 = compute_s_distribution(rho_thermal, x_eval, p_eval, s=0.0)
Q_th_sm1 = compute_s_distribution(rho_thermal, x_eval, p_eval, s=-1.0)

print(f"--- Thermal State Characteristic Function Benchmarking ---")
print(f"Thermal Wigner (s=0) Center Value: {W_th_s0[40, 40]:.5f} | Expected: {1.0/(np.pi*(2*n_bar_val + 1)):.5f}")
print(f"Thermal Husimi (s=-1) Center Value: {Q_th_sm1[40, 40]:.5f} | Expected: {1.0/(2.0*np.pi*(n_bar_val + 1)):.5f}")
print(f"Wigner Integral over dx dp: {np.sum(W_th_s0)*(x_eval[1]-x_eval[0])**2:.5f}")
print(f"Husimi Integral over dx dp: {np.sum(Q_th_sm1)*(x_eval[1]-x_eval[0])**2:.5f}")


## 3. Singularity of $P(\alpha)$ and the Kiesel-Vogel Filtered $P_\Omega(\alpha)$

Why can't we compute $P(\alpha) = W(\alpha; s=+1)$ directly for a single-photon Fock state $|1\rangle$?

For a Fock state $|n\rangle$, the characteristic function is:
$$
\chi_n(\xi; 0) = e^{-|\xi|^2/2} L_n(|\xi|^2)
$$
Multiplying by $e^{+s|\xi|^2/2}$ with $s = +1$:
$$
\chi_n(\xi; +1) = e^{-|\xi|^2/2} L_n(|\xi|^2) \cdot e^{+|\xi|^2/2} = L_n(|\xi|^2)
$$
Notice the Gaussian damping $e^{-|\xi|^2/2}$ **completely vanishes**!
For Fock state $|1\rangle$, $L_1(|\xi|^2) = 1 - |\xi|^2$.
Taking the Fourier transform of a polynomial yields **derivatives of the Dirac delta function**:
$$
P_1(\alpha) = \left( 1 + \frac{\partial^2}{\partial \alpha \partial \alpha^*} \right) \delta^{(2)}(\alpha)
$$
This is a **tempered distribution** with infinitely negative singularities! It cannot be plotted on a computer or measured directly by an analog detector.

---

### The Kiesel-Vogel Regularization Filter (2008–2010)

To overcome this singularity and make non-classical $P$-distributions directly measurable, **Jan Kiesel and Werner Vogel** (*Phys. Rev. A* 77, 062106, 2008; *Phys. Rev. Lett.* 105, 123603, 2010) introduced the **regularized $P$-function**:

$$
P_\Omega(\alpha) \equiv \frac{1}{\pi^2} \int_{\mathbb{C}} \chi(\xi; +1) \, \Omega_w(\xi) \, e^{\alpha \xi^* - \alpha^* \xi} \, d^2\xi
$$

where $\Omega_w(\xi)$ is an apodized low-pass filter with cutoff width $w$:
$$
\Omega_w(\xi) = \exp\left( -w |\xi|^2 \right) \quad \text{or} \quad \Omega_w(\xi) = \left[ 1 - \frac{|\xi|^2}{w^2} \right]^2 \Theta(w - |\xi|)
$$

### Crucial Theorem:
1. For any classical state ($P(\alpha) \ge 0$), the filtered distribution $P_\Omega(\alpha) \ge 0$ is strictly positive for all filter widths $w$.
2. **If $P_\Omega(\alpha) < 0$ for any filter bandwidth, the state is unconditionally non-classical!**


In [ ]:
# Simulating the Kiesel-Vogel Filtered P-Function P_Omega(alpha) for Fock |1>

# For Fock state |1>, chi(xi; s=1) = 1 - |xi|^2
# Filtered characteristic function: chi_filtered(xi) = (1 - |xi|^2) * exp(-w * |xi|^2)
# Here w acts as an effective Gaussian filter width

w_filter_values = [0.45, 0.35, 0.25]  # Decreasing filter width approaches singular P
colors_filter = ['#1f77b4', '#ff7f0e', '#d62728']

r_pts = np.linspace(0, 3.5, 400)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: Radial profiles of Regularized P_Omega(r)
# Analytical Fourier transform of (1 - |xi|^2) * exp(-w * |xi|^2):
# Let u = |xi|^2. The 2D Fourier transform yields:
# P_Omega(r) = (1 / (pi * w^2)) * exp(-r^2 / w) * [ (w - 1)/w + r^2 / w^2 ]
for w_val, col in zip(w_filter_values, colors_filter):
    # Analytical regularized profile
    p_omega_r = (1.0 / (np.pi * w_val**2)) * np.exp(-r_pts**2 / w_val) * ((w_val - 1.0)/w_val + (r_pts**2)/(w_val**2))
    ax1.plot(r_pts, p_omega_r, color=col, lw=2.2,
             label=rf'Filter $w = {w_val:.2f}$ (Min $P_\Omega = {p_omega_r[0]:.2f}$)')

ax1.axhline(0.0, color='black', lw=1.2, ls='--')
ax1.set_xlim(0, 3.0)
ax1.set_ylim(-1.6, 2.5)
ax1.set_xlabel(r"Phase Space Radius $r = |\alpha|$")
ax1.set_ylabel(r"Regularized $P_\Omega(r)$")
ax1.set_title(r"Kiesel-Vogel $P_\Omega(r)$: Deepening Negative Singularity for Fock $|1\rangle$")
ax1.legend(loc='upper right', framealpha=0.92)
ax1.grid(True)

# Panel 2: 2D Contour Map for w = 0.35
X_p, P_p = np.meshgrid(np.linspace(-2.5, 2.5, 101), np.linspace(-2.5, 2.5, 101))
R2_p = X_p**2 + P_p**2
w_plot = 0.35
P_2D = (1.0 / (np.pi * w_plot**2)) * np.exp(-R2_p / w_plot) * ((w_plot - 1.0)/w_plot + R2_p / (w_plot**2))

cf = ax2.contourf(X_p, P_p, P_2D, levels=np.linspace(-1.2, 1.8, 45), cmap='RdBu_r', extend='both')
ax2.contour(X_p, P_p, P_2D, levels=[0.0], colors='black', linewidths=2.0, linestyles='--')
ax2.set_aspect('equal')
ax2.set_xlim(-2.5, 2.5)
ax2.set_ylim(-2.5, 2.5)
ax2.set_xlabel(r"$\mathrm{Re}(\alpha)$")
ax2.set_ylabel(r"$\mathrm{Im}(\alpha)$")
ax2.set_title(rf"2D Regularized $P_\Omega(\alpha)$ ($w={w_plot:.2f}$, Negative Core in Blue)")
plt.colorbar(cf, ax=ax2, fraction=0.046, pad=0.04, label=r'$P_\Omega(\alpha)$')

plt.tight_layout()
plt.show()

print(f"Kiesel-Vogel filtered P-function verified: negative core confirmed for single-photon state!")


## 4. The Triad of Phase-Space Representations

We have now explored all three pillars of continuous quantum phase space:

```
           Glauber-Sudarshan P(α)     [s = +1: Normal Ordering]
                     │
                     │  Convolve with Vacuum Gaussian (σ² = 1/2)
                     ▼
              Wigner W(α)             [s =  0: Symmetric Weyl Ordering]
                     │
                     │  Convolve with Vacuum Gaussian (σ² = 1/2)
                     ▼
               Husimi Q(α)            [s = -1: Anti-Normal Ordering]
```

---

### Master Comparison Table: $P(\alpha)$, $W(\alpha)$, and $Q(\alpha)$

| Property | Glauber-Sudarshan $P(\alpha)$ | Wigner $W(\alpha)$ | Husimi $Q(\alpha)$ |
| :--- | :--- | :--- | :--- |
| **Ordering Parameter** | $s = +1$ | $s = 0$ | $s = -1$ |
| **Operator Ordering** | Normal ($:\hat{a}^\dagger \hat{a}:$) | Symmetric (Weyl) | Anti-Normal ($::\hat{a} \hat{a}^\dagger::$) |
| **Coherent State $|\alpha_0\rangle$** | $\delta^{(2)}(\alpha - \alpha_0)$ | $\frac{1}{\pi}e^{-2|\alpha-\alpha_0|^2}$ | $\frac{1}{\pi}e^{-|\alpha-\alpha_0|^2}$ |
| **Single-Photon State $|1\rangle$** | $(1 + \partial^2/\partial\alpha\partial\alpha^*)\delta(\alpha)$ | $\frac{1}{\pi}e^{-2|\alpha|^2}(4|\alpha|^2 - 1)$ | $\frac{1}{\pi}|\alpha|^2 e^{-|\alpha|^2}$ |
| **Thermal State $\hat{\rho}_{\text{th}}$** | $\frac{1}{\pi\bar{n}}e^{-|\alpha|^2/\bar{n}}$ | $\frac{1}{\pi(2\bar{n}+1)}e^{-\frac{2|\alpha|^2}{2\bar{n}+1}}$ | $\frac{1}{\pi(\bar{n}+1)}e^{-\frac{|\alpha|^2}{\bar{n}+1}}$ |
| **Negativity Behavior** | Negative singular distributions | Negative smooth lobes ($W < 0$) | **Strictly non-negative ($Q \ge 0$)** |
| **Experimental Access** | Regularized filter $P_\Omega$ | Homodyne Tomography (Radon) | 8-Port Homodyne (Heterodyne) |


In [ ]:
# Comparative Visual Triad: Filtered P, Wigner W, and Husimi Q for Single Photon

# Compute Wigner W(alpha) and Husimi Q(alpha) on identical grid
x_triad = np.linspace(-3.0, 3.0, 91)
p_triad = np.linspace(-3.0, 3.0, 91)
X_t, P_t = np.meshgrid(x_triad, p_triad)
r2_t = X_t**2 + P_t**2

# Analytical distributions for Fock |1>:
# 1. Regularized P_Omega (w = 0.4)
w_t = 0.4
P_triad = (1.0 / (np.pi * w_t**2)) * np.exp(-r2_t / w_t) * ((w_t - 1.0)/w_t + r2_t / (w_t**2))

# 2. Wigner W(x, p)
W_triad = (1.0 / np.pi) * np.exp(-r2_t) * (2.0 * r2_t - 1.0)

# 3. Husimi Q(x, p)
Q_triad = (1.0 / (4.0 * np.pi)) * r2_t * np.exp(-0.5 * r2_t)

# Plotting: 3-panel comparative layout
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Filtered P_Omega
cf1 = ax1.contourf(x_triad, p_triad, P_triad, levels=np.linspace(-0.8, 1.2, 40), cmap='RdBu_r', extend='both')
ax1.contour(x_triad, p_triad, P_triad, levels=[0.0], colors='black', linewidths=1.8, linestyles='--')
ax1.set_title(r"Glauber $P_\Omega(\alpha)$ ($s=+1$ Filtered)")
plt.colorbar(cf1, ax=ax1, fraction=0.046, pad=0.04)

# Panel 2: Wigner W(x, p)
cf2 = ax2.contourf(x_triad, p_triad, W_triad, levels=np.linspace(-0.35, 0.35, 40), cmap='RdBu_r', extend='both')
ax2.contour(x_triad, p_triad, W_triad, levels=[0.0], colors='black', linewidths=1.8, linestyles='--')
ax2.set_title(r"Wigner $W(x, p)$ ($s=0$ Symmetric)")
plt.colorbar(cf2, ax=ax2, fraction=0.046, pad=0.04)

# Panel 3: Husimi Q(x, p)
cf3 = ax3.contourf(x_triad, p_triad, Q_triad, levels=np.linspace(0, 0.12, 40), cmap='viridis')
ax3.set_title(r"Husimi $Q(x, p)$ ($s=-1$ Anti-Normal)")
plt.colorbar(cf3, ax=ax3, fraction=0.046, pad=0.04)

for ax in [ax1, ax2, ax3]:
    ax.set_aspect('equal')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_xlabel(r"$\mathrm{Re}(\alpha)$")
    ax.set_ylabel(r"$\mathrm{Im}(\alpha)$")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Interactive Exploration: Dynamic Sweep of Ordering Parameter s in [-1, +1]

def explore_ordering_sweep(s_param=0.0, state_choice='Thermal (n=1.5)'):
    """Interactive slider smoothly tuning ordering parameter s from -1 (Q) to 0 (W)."""
    N_s = 20
    x_s = np.linspace(-3.5, 3.5, 61)
    p_s = np.linspace(-3.5, 3.5, 61)
    
    if state_choice == 'Thermal (n=1.5)':
        n_b = 1.5
        rho = np.diag((1.0 / (1.0 + n_b)) * ((n_b / (1.0 + n_b))**np.arange(N_s))).astype(np.complex128)
    elif state_choice == 'Coherent (alpha=1.5)':
        a = np.diag(np.sqrt(np.arange(1, N_s)), k=1)
        D = la.expm(1.5 * a.T.conj() - 1.5 * a)
        vac = np.zeros(N_s, dtype=np.complex128); vac[0] = 1.0
        psi = D @ vac
        rho = np.outer(psi, psi.conj())
    else:  # Squeezed
        a = np.diag(np.sqrt(np.arange(1, N_s)), k=1)
        S = la.expm(0.5 * 0.6 * (a@a - a.T.conj()@a.T.conj()))
        vac = np.zeros(N_s, dtype=np.complex128); vac[0] = 1.0
        psi = S @ vac
        rho = np.outer(psi, psi.conj())
        
    Dist = compute_s_distribution(rho, x_s, p_s, s=s_param)
    min_v = np.min(Dist)
    max_v = np.max(Dist)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # 2D contour
    v_bound = max(0.2, max_v)
    cf = ax1.contourf(x_s, p_s, Dist, levels=np.linspace(min(0, min_v), v_bound, 40), cmap='plasma')
    ax1.set_aspect('equal')
    ax1.set_xlabel("x")
    ax1.set_ylabel("p")
    ax1.set_title(rf"$W(\alpha; s={s_param:+.2f})$ for {state_choice}")
    plt.colorbar(cf, ax=ax1, fraction=0.046, pad=0.04)
    
    # 1D slice along x
    mid_idx = len(p_s) // 2
    slice_1d = Dist[mid_idx, :]
    ax2.plot(x_s, slice_1d, 'r-', lw=2.2, label=rf'Slice at $p=0$ (Peak={max_v:.3f})')
    ax2.set_xlim(-3.5, 3.5)
    ax2.set_xlabel("x")
    ax2.set_ylabel("Amplitude")
    ax2.set_title(rf"Phase Space Spread: Width grows as $s \to -1$ (Anti-normal)")
    ax2.grid(True)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

# Launch interactive exploration
if HAS_WIDGETS:
    interact(explore_ordering_sweep,
             s_param=FloatSlider(min=-1.0, max=0.0, step=0.1, value=0.0, description='Ordering s:'),
             state_choice=Dropdown(options=['Thermal (n=1.5)', 'Coherent (alpha=1.5)', 'Squeezed'], value='Thermal (n=1.5)'))
else:
    # Static execution fallback for headless environments
    explore_ordering_sweep(s_param=0.0, state_choice='Thermal (n=1.5)')


## 5. Self-Check Exercises and Pedagogical Challenges

Test your understanding of the Glauber-Sudarshan $P$-representation and operator orderings.

---

### Challenge 1: Proof that Coherent States Have Dirac Delta $P$-Functions
Given the definition:
$$
\hat{\rho} = \int P(\alpha) |\alpha\rangle\langle \alpha| \, d^2\alpha
$$
1. Prove that for a pure coherent state $\hat{\rho} = |\alpha_0\rangle\langle \alpha_0|$, the unique $P$-function is $P(\alpha) = \delta^{(2)}(\alpha - \alpha_0)$.
2. Calculate the average photon number $\langle \hat{n} \rangle$ using the $P$-function integral formula $\langle \hat{a}^\dagger \hat{a} \rangle = \int P(\alpha) |\alpha|^2 \, d^2\alpha$ and verify that it equals $|\alpha_0|^2$.

<details>
<summary><b>Click to expand Solution to Challenge 1</b></summary>

**Part 1 Solution:**
Substitute $P(\alpha) = \delta^{(2)}(\alpha - \alpha_0)$ into the definition of $\hat{\rho}$:
$$
\hat{\rho} = \int \delta^{(2)}(\alpha - \alpha_0) |\alpha\rangle\langle \alpha| \, d^2\alpha = |\alpha_0\rangle\langle \alpha_0|
$$
By inspection, the integral reproduces the exact coherent density projector. Since the $P$-representation is unique within the space of tempered distributions, $P(\alpha) = \delta^{(2)}(\alpha - \alpha_0)$.

**Part 2 Solution:**
Using the normal-ordered moment formula:
$$
\langle \hat{a}^\dagger \hat{a} \rangle = \int \delta^{(2)}(\alpha - \alpha_0) |\alpha|^2 \, d^2\alpha = |\alpha_0|^2
$$
This confirms that the coherent state represents a deterministic point in phase space with zero intensity variance beyond vacuum shot noise.
</details>

---

### Challenge 2: Numerical Verification of Thermal $P$-Function
Verify that the numerical $s$-parameterized solver with $s = 0$ reproduces the exact analytical Wigner variance for a thermal state $\sigma_W^2 = \bar{n} + 1/2$.

<details>
<summary><b>Click to expand Solution to Challenge 2</b></summary>

```python
# Verification of Thermal Wigner Variance
n_bar = 2.0
# Exact analytical: W(x, p) = 1/(pi*(2*n_bar+1)) * exp(-(x^2+p^2)/(2*n_bar+1))
# Variance: sigma_x^2 = (2*n_bar + 1)/2 = n_bar + 0.5 = 2.5
exact_var = n_bar + 0.5
print(f"Analytical Wigner Variance: {exact_var:.4f}")

# From numerical W_th_s0:
x_ax = np.linspace(-4, 4, 81)
p_x = np.sum(W_th_s0, axis=0) * (x_ax[1] - x_ax[0])
num_var = np.sum(x_ax**2 * p_x) * (x_ax[1] - x_ax[0])
print(f"Numerical Marginal Variance: {num_var:.4f}")
assert abs(num_var - exact_var) < 0.05, "Thermal variance verification failed!"
print("Thermal state variance benchmark passed!")
```
</details>


## 6. Summary, Reference Table, and Outlook

### Comprehensive Synthesis: The Operator Ordering Landscape

| Ordering | Parameter $s$ | Quasiprobability | Mathematical Character | Physical Measurement |
| :--- | :---: | :--- | :--- | :--- |
| **Normal** | $s = +1$ | **Glauber-Sudarshan $P(\alpha)$** | Singular (derivatives of $\delta$) | Kiesel-Vogel regularized $P_\Omega$ |
| **Symmetric** | $s = 0$ | **Wigner $W(x, p)$** | Smooth, real, negative lobes | Balanced Homodyne Tomography |
| **Anti-Normal** | $s = -1$ | **Husimi $Q(x, p)$** | Smooth, strictly non-negative | 8-Port Homodyne (Heterodyne) |

---

### Key Takeaways
1. **The Classicality Boundary:** $P(\alpha) \ge 0$ is the definitive mathematical line separating classical electromagnetic mixtures from non-classical quantum light.
2. **Singularity as Quantumness:** Non-classical states exhibit singular $P$-functions with negative infinities; the Kiesel-Vogel filter regularizes these distributions into observable non-classical witnesses.
3. **The Smoothing Chain:** $P \to W \to Q$ corresponds to successive convolutions with vacuum noise Gaussians, each step trading quantum phase-space resolution for mathematical smoothness.

---

### Primary References
- **Glauber, R. J.** (1963). *Coherent and incoherent states of the radiation field*. Physical Review, 131(6), 2766–2788.
- **Sudarshan, E. C. G.** (1963). *Equivalence of semiclassical and quantum mechanical descriptions of statistical light beams*. Physical Review Letters, 10(7), 277–279.
- **Cahill, K. E., & Glauber, R. J.** (1969). *Ordered expansions in boson amplitude operators*. Physical Review, 177(5), 1857–1881.
- **Kiesel, J., & Vogel, W.** (2010). *Nonclassicality filters and quasiprobabilities*. Physical Review Letters, 105(12), 123603.

---

### Next in the Curriculum
In **Notebook 2.5: Schrödinger Cat States and Phase Space Interference**, we investigate macroscopic quantum superpositions $|\alpha\rangle \pm |-\alpha\rangle$, discovering sub-Planck phase-space structures, ultra-sensitive phase estimation, and environmental decoherence.
